## Paso 1 — Cargamos las 18 tablas ya tipadas

Mismo patrón que el notebook 01: leemos directo los Parquet del notebook
00, sin depender de que otro notebook se haya corrido antes en esta misma
sesión.


In [1]:
import pandas as pd
from pathlib import Path

RUTA_INTERIM = Path("../datos/interim")
ARCHIVOS_PARQUET = sorted(RUTA_INTERIM.glob("*.parquet"))

tablas = {archivo.stem: pd.read_parquet(archivo) for archivo in ARCHIVOS_PARQUET}

assert len(tablas) == 18, f"Se esperaban 18 tablas, se cargaron {len(tablas)}"
print(f"Tablas cargadas: {len(tablas)}")


Tablas cargadas: 18


## Paso 2 — Duplicados de fila completa y unicidad de Id

Por cada una de las 18 tablas: ¿hay filas completamente duplicadas?, y
donde exista una columna `id`, ¿identifica de verdad una sola fila cada
valor? A diferencia del EDA-1 (donde `FO` resultó no ser clave única), aquí
se espera que sí lo sea — son GUID/identificadores de sistema — pero se
confirma con código, no se asume.


In [2]:
resumen_unicidad = []
for nombre, df in tablas.items():
    n_filas = len(df)
    n_duplicados_completos = int(df.duplicated().sum())
    columna_id = "id" if "id" in df.columns else None
    if columna_id:
        n_ids_unicos = df[columna_id].nunique()
        id_es_unico = n_ids_unicos == n_filas
    else:
        n_ids_unicos = None
        id_es_unico = None
    resumen_unicidad.append({
        "tabla": nombre,
        "filas": n_filas,
        "duplicados_fila_completa": n_duplicados_completos,
        "tiene_columna_id": columna_id is not None,
        "ids_unicos": n_ids_unicos,
        "id_es_clave_unica": id_es_unico,
    })

df_resumen_unicidad = pd.DataFrame(resumen_unicidad)
df_resumen_unicidad


,tabla,filas,duplicados_fila_completa,tiene_columna_id,ids_unicos,id_es_clave_unica
0,audit_log,72,0,True,72.0,True
1,catalogo-clientes-BUT000-organizaciones,38995,0,False,NaN,None
2,catalogo_clientes,21,0,True,21.0,True
3,catalogo_configuraciones_vehiculo,16,0,True,16.0,True
4,catalogo_rutas,31,0,True,31.0,True
5,catalogo_servicios_conexos,11,0,True,11.0,True
6,catalogo_usuarios,11,0,True,11.0,True
7,cotizacion_conexos,7,0,True,7.0,True
8,cotizacion_lineas,48,0,True,48.0,True
9,cotizaciones,18,0,True,18.0,True


## Paso 3 — Huérfanos e integridad referencial (§4.2 del plan)

Por cada relación padre-hijo documentada, ¿toda llave foránea (`xxxId`)
en la tabla hija existe de verdad en la tabla padre? Se deja fuera
`audit_log.entidadid` a propósito — es una llave "polimórfica" (apunta a
distintas tablas según el valor de `entidad`), necesita un tratamiento
aparte que no corresponde a este chequeo genérico.


In [3]:
RELACIONES = [
    ("lineas_solicitud", "solicitudId", "solicitudes", "id"),
    ("lineas_solicitud", "rutaId", "catalogo_rutas", "id"),
    ("lineas_solicitud", "configVehiculoId", "catalogo_configuraciones_vehiculo", "id"),
    ("cotizacion_lineas", "cotizacionId", "cotizaciones", "id"),
    ("cotizacion_lineas", "lineaSolicitudId", "lineas_solicitud", "id"),
    ("cotizacion_conexos", "cotizacionId", "cotizaciones", "id"),
    ("cotizacion_conexos", "servicioConexoId", "catalogo_servicios_conexos", "id"),
    ("cotizaciones", "solicitudId", "solicitudes", "id"),
    ("cotizaciones", "entregadaPor", "catalogo_usuarios", "id"),
    ("solicitudes", "clienteId", "catalogo_clientes", "id"),
    ("solicitudes", "creadoPor", "catalogo_usuarios", "id"),
    ("tarifas_historicas", "rutaId", "catalogo_rutas", "id"),
    ("tarifas_historicas", "configVehiculoId", "catalogo_configuraciones_vehiculo", "id"),
    ("tarifas_historicas", "cotizacionId", "cotizaciones", "id"),
]

resumen_referencial = []
for tabla_hija, columna_fk, tabla_padre, columna_pk in RELACIONES:
    df_hija = tablas[tabla_hija]
    df_padre = tablas[tabla_padre]
    valores_padre = set(df_padre[columna_pk].dropna().unique())

    poblado = df_hija[columna_fk].notna()
    n_poblados = int(poblado.sum())
    valores_fk = df_hija.loc[poblado, columna_fk]
    huerfanos = ~valores_fk.isin(valores_padre)
    n_huerfanos = int(huerfanos.sum())

    resumen_referencial.append({
        "tabla_hija": tabla_hija,
        "columna_fk": columna_fk,
        "tabla_padre": tabla_padre,
        "filas_hija": len(df_hija),
        "fk_poblados": n_poblados,
        "huerfanos": n_huerfanos,
        "pct_huerfanos_de_poblados": round(n_huerfanos / n_poblados * 100, 1) if n_poblados else None,
    })

df_resumen_referencial = pd.DataFrame(resumen_referencial)
df_resumen_referencial


,tabla_hija,columna_fk,tabla_padre,filas_hija,fk_poblados,huerfanos,pct_huerfanos_de_poblados
0,lineas_solicitud,solicitudId,solicitudes,43,43,0,0.0
1,lineas_solicitud,rutaId,catalogo_rutas,43,43,0,0.0
2,lineas_solicitud,configVehiculoId,catalogo_configuraciones_vehiculo,43,8,0,0.0
3,cotizacion_lineas,cotizacionId,cotizaciones,48,48,0,0.0
4,cotizacion_lineas,lineaSolicitudId,lineas_solicitud,48,48,0,0.0
5,cotizacion_conexos,cotizacionId,cotizaciones,7,7,0,0.0
6,cotizacion_conexos,servicioConexoId,catalogo_servicios_conexos,7,7,0,0.0
7,cotizaciones,solicitudId,solicitudes,18,18,5,27.8
8,cotizaciones,entregadaPor,catalogo_usuarios,18,6,0,0.0
9,solicitudes,clienteId,catalogo_clientes,13,13,0,0.0


## Paso 4 — Grano de historico-sap: ¿cuántas versiones por tarifa?

`historico-sap` (tblSAPTarifasLog) guarda una fila por cada versión
histórica de una tarifa, enlazada a `tarifa-plaza` por `idTarifa`. El
diccionario deja abierto si esto es "auditoría de cada edición" o
"histórico de tarifas realmente aplicadas" (pendiente de confirmar con
negocio). No podemos resolver esa pregunta con el dato solo, pero sí medir
cuántas versiones tiene en promedio cada tarifa — un indicio indirecto.


In [4]:
tabla_historico = tablas["historico-sap-tblSAPTarifasLog-2022-09-01_a_hoy"]
tabla_tarifa_plaza = tablas["tarifa-plaza-tblSAPTarifas-2022-09-01_a_hoy"]

versiones_por_tarifa = tabla_historico.groupby("idTarifa").size()

print(f"Tarifas con al menos 1 versión en el histórico: {versiones_por_tarifa.shape[0]:,}")
print(f"Total de tarifas en tarifa-plaza (vigente): {len(tabla_tarifa_plaza):,}")
print()
print("Distribución de versiones por tarifa:")
print(versiones_por_tarifa.describe())
print()
print("Percentiles altos (para ver si hay tarifas con muchísimas versiones):")
print(versiones_por_tarifa.quantile([0.5, 0.75, 0.9, 0.95, 0.99, 1.0]))


Tarifas con al menos 1 versión en el histórico: 8,135
Total de tarifas en tarifa-plaza (vigente): 23,020

Distribución de versiones por tarifa:
count    8135.000000
mean        3.112231
std         3.485834
min         1.000000
25%         1.000000
50%         2.000000
75%         4.000000
max        53.000000
dtype: float64

Percentiles altos (para ver si hay tarifas con muchísimas versiones):
0.50     2.0
0.75     4.0
0.90     7.0
0.95    10.0
0.99    17.0
1.00    53.0
dtype: float64


## Síntesis del notebook 02 — Unicidad y granularidad (Fase 3, §4 del plan)

**Objetivo cumplido:** confirmar que cada fila representa lo que dice
representar, y que las relaciones padre-hijo documentadas no tienen
huérfanos silenciosos.

**Duplicados y unicidad de Id (Paso 2):** 0 filas duplicadas en las 18
tablas. 15 de 16 tablas con columna `id` la tienen como clave única
perfecta — a diferencia del `FO` del EDA-1, aquí no hubo sorpresa grande.
La única excepción, `piso-sicetac_raras` (4 `id` repetidos de 25), tiene
causa confirmada: son 4 rutas reales que califican como extremas en 2
categorías a la vez (valor máximo **y** vía no estándar, simultáneamente)
— no es un error, es una fila que ganó dos "medallas" al mismo tiempo.
Verificado además que `nucleo` y `raras` no comparten ningún `id` entre sí
(0 de intersección) — son mutuamente excluyentes como se esperaba.

**Huérfanos e integridad referencial (Paso 3):** 12 de 14 relaciones
padre-hijo perfectas (0 huérfanos). Dos no: `cotizaciones.solicitudId`
(5 de 18, todas en estado `Borrador`) y `tarifasHistoricas.cotizacionId`
(1 sola cotización faltante, referenciada 3 veces). Causa no confirmada
por el usuario — consistente con que CabinaFletes todavía está en
desarrollo (no es un bug de este pipeline de EDA). Se documenta como
limitación con causa no confirmada, no como hallazgo cerrado.

**Grano de historico-sap (Paso 4):** solo el 35% de las tarifas vigentes
(8.135 de 23.020) tiene alguna versión en el histórico; de esas, la
mediana es 2 versiones. El patrón (pocas ediciones, la mayoría de tarifas
vigentes nunca se toca) es más consistente con "log de ediciones
ocasionales" que con "registro de cada vez que se aplicó la tarifa" —
evidencia indirecta a favor de una de las dos hipótesis del diccionario,
no una confirmación definitiva (sigue pendiente de negocio).

**Pista lateral, no resuelta aquí:** durante esta fase se descubrieron
tablas adicionales en la base SICE-TAC (`tblRutasSicetacMesAnterior`,
`MesAnterior2`, `ComparativoMesAnterior`) que podrían dar profundidad
histórica real a SICE-TAC — hoy solo tiene una foto de un período. Una
verificación preliminar fue inconclusa (la llave usada para comparar no
capturaba el grano completo de SICE-TAC). Queda anotada para retomar en
la Fase 8 (temporal/geográfico), con la llave completa
(`nombreRuta+configuracion+tipoCarga+condicionCarga+unidadTransporte`).

**Qué le queda listo al siguiente notebook:** confirmación de que el
grano de cada tabla es confiable, y 2 limitaciones puntuales (huérfanos,
grano de historico-sap con causa no confirmada) para `limitaciones.md`.
`03_calidad_del_dato.ipynb` (Fase 4, §5) parte de aquí.
